# DATA EXTRACTION AND CLEANING

## Text Data

### Data loading steps

In [ ]:
import pandas as pd
import numpy as np

all_datas =["all_speeches_90_22.csv","communications_20_27.csv","CrudeOilNewsData_26.csv","geopolitical_events_timeline.csv",
             "gold-dataset-sinha-khandait_00_19.csv","guardian_environment_news_17_24.csv",
             "Newsdata_Records_2026_08_27_08_18_00.csv","nyt_2022_2025.csv","sp500_headlines_2008_2024.csv"]
all_datas = ["../../data/"+k for k in all_datas]

#all_datas was used to upload the crude data 

dfs =[]
col_all = ["date","text"]
col_com = ["Date","Text"]
col_Crude = ["publishedAt","description"]
col_geop = ["date","event_description"]
col_gold = ["Dates","News"]
col_guard = ["Date Published","Article Text"]
col_News = ["pubDate","ai_summary"]
col_nyt = ["pub_date","lead_paragraph"]
col_sp500 = ["Date","Title"]
cols =[col_all,col_com,col_Crude,col_geop,col_gold,col_guard,col_News,col_nyt,col_sp500]

# Load all the datas
for data in all_datas:
    dfs.append(pd.read_csv(data))



In [4]:


numb_data = len(dfs)

# Take the right colums
for i in range(numb_data) :
    dfs[i] = dfs[i][cols[i]] # only dates and texts
    dfs[i] = dfs[i].dropna() # dropping any row with missing value (nan or None)
    dfs[i] = dfs[i].drop_duplicates() # Drop any duplicated row
    dfs[i] = dfs[i].rename(columns = {cols[i][0]:"DATES",cols[i][1]:"TEXT"}) #Rename all the columns of the data
    



In [5]:
# 

dfs_weekly = []
for i in range(numb_data):
    df = dfs[i].copy()
    d = pd.to_datetime(df["DATES"], utc=True, format="mixed", errors="coerce").dt.tz_localize(None)
    df["week"] = d.dt.to_period("W-FRI").dt.end_time.dt.normalize()
    df = (df.groupby("week")
            .agg(TEXT=("TEXT", lambda s: " || ".join(s.astype(str))),
                 n_articles=("TEXT", "size"))
            .reset_index())
    dfs_weekly.append(df)

# Merge the data into one

data = pd.concat(dfs_weekly, ignore_index=True)        # 1. put all DataFrames together

data = (data.groupby("week")                    # 2. one row per week
            .agg(TEXT=("TEXT", " || ".join),
                 n_articles=("n_articles", "sum"))
            .reset_index())

data = data[(data["week"] >= "2010-01-01") & (data["week"] <= "2026-12-31")]

### Using LLM to score the texts

In [ ]:
#define the answer format

from typing import List, Literal
from pydantic import BaseModel
from skfin.llm  import apply_prompt

class CommodityImpact(BaseModel):
    commodity: Literal["crude_oil", "natural_gas", "gold", "copper", "wheat"]
    impact: float   # -1 (very bearish) to +1 (very bullish), 0 = no effect

class Impacts(BaseModel):
    impacts: List[CommodityImpact]

In [7]:
#The prompt

def commodity_prompt(text):
    return f"""Read the news text below. For EACH of these five commodities:
crude oil, natural gas, gold, copper, wheat, estimate the likely impact of this news
on its futures price over the next week, as a number between -1 (strongly bearish)
and +1 (strongly bullish). Use 0 if the news is irrelevant to that commodity.
Always return all five commodities.

Text: {text}"""

In [ ]:
# Install and get the key (you need to paste your key) since i've used mine.
#Ensure you get enough credits on open.ai site

import os
#os.environ["OPENAI_API_KEY"] = "sk-..."    # your key, in a cell you don't share
api_key = os.environ.get("OPENAI_API_KEY")

sk-proj- 164


####  Test the LLM function with a small data

In [ ]:


articles = pd.concat(dfs, ignore_index=True)        # all sources, one row per article

d = pd.to_datetime(articles["DATES"], utc=True, format="mixed", errors="coerce").dt.tz_localize(None)
articles["week"] = d.dt.to_period("W-FRI").dt.end_time.dt.normalize()
articles = articles[(articles["week"] >= "2010-01-01") & (articles["week"] <= "2025-12-31")]
keywords = ["federal reserve", "interest rate", "inflation", "budget", "deficit", "recession",
            "crude", "opec", "oil price", "natural gas", "gold price", "copper", "wheat"]
articles = articles[articles["TEXT"].str.lower().str.contains("|".join(keywords), regex=True)]
articles = articles.reset_index(drop=True)
print(len(articles))


11806


In [ ]:
# Test to ensure that the llm function works
sample = articles.head(20).copy()
sample["TEXT"] = sample["TEXT"].str[:2000]          # cap length to control cost
scores = apply_prompt(sample, content_function=commodity_prompt,
                      response_format=Impacts, nested_response=True,
                      text_column="TEXT", sequential=True)
print(sample[["TEXT"]].join(scores))

                                                 TEXT  crude_oil  natural_gas  \
0   Since we last met the global economy has conti...        0.5          0.0   
1   Welcome back to Sydney. The last time that thi...        0.0          0.0   
2   Thank you for the invitation to visit Toowoomb...        0.0          0.0   
3   Thank you for the invitation to be here today ...        0.0          0.0   
4   Thank you for coming along today in support of...        0.0          0.0   
5   Thank you for the invitation to deliver the 20...        0.0          0.0   
6   Thank you for the invitation to come to Sheppa...        0.0          0.0   
7   The topic the AIG had proposed for this sessio...        0.0          0.0   
8   When we last met with the Committee in Februar...        0.5          0.0   
9   Thank you for the invitation to play a part in...        0.0          0.0   
10  Hansard transcripts of public hearings are mad...        0.0          0.0   
11  The rise in prices for n

#### Back to our data

In [10]:
articles = data[["week", "TEXT"]].copy()
articles["TEXT"] = articles["TEXT"].str.split(" || ", regex=False)   # one list per week
articles = articles.explode("TEXT", ignore_index=True)               # one row per article
articles = articles[articles["TEXT"].str.len() > 30]                 # drop empty bits

keywords = ["federal reserve", "interest rate", "inflation", "budget", "deficit", "recession",
            "crude", "opec", "oil price", "natural gas", "gold price", "copper", "wheat"]
articles = articles[articles["TEXT"].str.lower().str.contains("|".join(keywords), regex=False) == False] if False else \
           articles[articles["TEXT"].str.lower().str.contains("|".join(keywords))]
articles = articles.reset_index(drop=True)
articles.to_csv("articles.csv", index=False)
print(len(articles))


15498


In [ ]:
#Applying the llm function

for start in range(0, len(articles), 500):
    path = f"scores_{start}.csv"
    if os.path.exists(path):
        continue
    chunk = articles.iloc[start:start + 500].copy()
    chunk["TEXT"] = chunk["TEXT"].str[:2000]
    out = apply_prompt(chunk, content_function=commodity_prompt, response_format=Impacts,
                   nested_response=True, text_column="TEXT", max_workers=10,
                   api_key=api_key)
    chunk.join(out).to_csv(path, index=False)

In [ ]:
# Creating the scored data from the llm

import glob
names = ["crude_oil", "natural_gas", "gold", "copper", "wheat"]
scored = pd.concat([pd.read_csv(f) for f in glob.glob("scores_*.csv")])
scored["week"] = pd.to_datetime(scored["week"])

relevant = scored[names].where(scored[names] != 0)                   # 0 = irrelevant -> ignored
weekly_tone = relevant.groupby(scored["week"]).mean().fillna(0)      # average impact per week
weekly_count = relevant.notna().groupby(scored["week"]).sum()        # number of relevant articles

import glob
names = ["crude_oil", "natural_gas", "gold", "copper", "wheat"]

scored = pd.concat([pd.read_csv(f) for f in glob.glob("scores_*.csv")], ignore_index=True)
scored = scored.drop_duplicates(subset=["week", "TEXT"])          # in case a block was saved twice
scored["week"] = pd.to_datetime(scored["week"])

relevant = scored[names].where(scored[names] != 0)                # 0 = irrelevant -> ignored
relevant["week"] = scored["week"]

weekly_tone = relevant.groupby("week")[names].mean().fillna(0)    # average impact per week
weekly_count = relevant.groupby("week")[names].count()            # number of relevant articles

weekly = weekly_tone.join(weekly_count, rsuffix="_n").reset_index()
print(weekly["week"].duplicated().sum())                          # should print 0

0


In [28]:

#weekly.head(10)
#weekly.shape #(811, 11)
print(weekly.describe())


                             week   crude_oil  natural_gas        gold  \
count                         811  811.000000   811.000000  811.000000   
mean   2017-11-26 15:23:18.273736   -0.167463    -0.093098    0.204784   
min           2010-01-01 00:00:00   -1.000000    -1.000000   -1.000000   
25%           2013-12-09 12:00:00   -0.500000    -0.168706    0.000000   
50%           2017-11-03 00:00:00    0.000000     0.000000    0.166667   
75%           2021-09-20 12:00:00    0.000000     0.000000    0.500000   
max           2026-09-18 00:00:00    1.000000     1.000000    1.000000   
std                           NaN    0.492213     0.435222    0.461825   

           copper       wheat  crude_oil_n  natural_gas_n      gold_n  \
count  811.000000  811.000000   811.000000     811.000000  811.000000   
mean     0.059985    0.097833     6.816276       2.186190    4.217016   
min     -1.000000   -1.000000     0.000000       0.000000    0.000000   
25%      0.000000    0.000000     0.00000

## Prices Data and Return

In [29]:
#download daily futures prices

import yfinance as yf

tickers = {"CL=F": "crude_oil", "NG=F": "natural_gas", "GC=F": "gold",
           "HG=F": "copper", "ZW=F": "wheat"}
end = weekly["week"].max() + pd.Timedelta(days=14)      # extra weeks for the last target

px = yf.download(list(tickers), start="2009-06-01", end=end,
                 auto_adjust=False, progress=False)["Close"]
px = px.rename(columns=tickers)
px.to_csv("futures_daily_close.csv")
print(px.tail())
print(px.isna().sum())
print(px.describe())

Ticker      crude_oil         gold  copper  natural_gas   wheat
Date                                                           
2026-09-25  92.410004  4321.200195  6.6955        3.196  703.25
2026-09-28  92.599998  4168.399902  6.5660        3.000  688.75
2026-09-29  89.379997  4179.700195  6.5435        3.011  692.75
2026-09-30  90.419998  4186.700195  6.5590        3.026  675.75
2026-10-01  92.870003  4202.299805  6.4820        2.967  682.75
Ticker
crude_oil      1
gold           1
copper         1
natural_gas    0
wheat          2
dtype: int64
Ticker    crude_oil         gold       copper  natural_gas        wheat
count   4362.000000  4362.000000  4362.000000  4363.000000  4361.000000
mean      72.081580  1770.124576     3.509142     3.395277   596.836104
std       20.367334   832.430286     0.938498     1.209537   136.147135
min      -37.630001   909.000000     1.939500     1.482000   361.000000
25%       56.462500  1259.950043     2.796000     2.647000   502.250000
50%       71.94

In [ ]:
#weekly prices and returns (Friday close)

wk = px.resample("W-FRI").last() #weekly prices (on friday)
wk.index = pd.to_datetime(wk.index).normalize()# without time
wk.index.name = "week"

ret = wk.pct_change(fill_method=None) #the weekly return

# s = ret.stack().sort_values() #Puts all returns of all commodities in one list
# print(s.head(3), s.tail(3))          # check the most extreme weekly moves
# ret = ret.clip(-0.5, 0.5)            # safety cap on absurd moves (roll jumps, data errors)

# s.describe()
# s.shape

DatetimeIndex(['2009-06-05', '2009-06-12', '2009-06-19', '2009-06-26',
               '2009-07-03', '2009-07-10', '2009-07-17', '2009-07-24',
               '2009-07-31', '2009-08-07',
               ...
               '2026-07-31', '2026-08-07', '2026-08-14', '2026-08-21',
               '2026-08-28', '2026-09-04', '2026-09-11', '2026-09-18',
               '2026-09-25', '2026-10-02'],
              dtype='datetime64[s]', name='week', length=905, freq='W-FRI')

In [ ]:
#price features and target

feats = pd.concat({
    "ret":     ret,                         # this week's return
    "mom_4w":  ret.rolling(4).sum(),        # momentum, 1 month
    "mom_12w": ret.rolling(12).sum(),       # momentum, 1 quarter
    "vol_12w": ret.rolling(12).std(),       # volatility, 1 quarter
    "target":  ret.shift(-1),               # NEXT week's return (what we predict)
}, axis=1)
feats.columns.names = ["feature", "commodity"]

price_panel = feats.stack(level="commodity", future_stack=True).reset_index()
price_panel.to_csv("price_features.csv", index=False)


feature                 week          ret       mom_4w      mom_12w  \
count                   4525  4520.000000  4505.000000  4465.000000   
mean     2018-02-02 00:00:00     0.001829     0.007480     0.022465   
min      2009-06-05 00:00:00    -0.293098    -0.763677    -1.022311   
25%      2013-10-04 00:00:00    -0.022136    -0.040478    -0.057813   
50%      2018-02-02 00:00:00     0.001234     0.005322     0.018439   
75%      2022-06-03 00:00:00     0.023860     0.053766     0.097590   
max      2026-10-02 00:00:00     0.699968     0.737780     0.955557   
std                      NaN     0.050354     0.096894     0.159484   

feature      vol_12w       target  
count    4465.000000  4520.000000  
mean        0.042514     0.001829  
min         0.007624    -0.293098  
25%         0.024395    -0.022136  
50%         0.035193     0.001234  
75%         0.051718     0.023860  
max         0.249565     0.699968  
std         0.027741     0.050354  


In [ ]:
#check how our data looks like 

print(price_panel.describe())
print(price_panel.head())
print(price_panel.columns)
print(price_panel.index)

feature                 week          ret       mom_4w      mom_12w  \
count                   4525  4520.000000  4505.000000  4465.000000   
mean     2018-02-02 00:00:00     0.001829     0.007480     0.022465   
min      2009-06-05 00:00:00    -0.293098    -0.763677    -1.022311   
25%      2013-10-04 00:00:00    -0.022136    -0.040478    -0.057813   
50%      2018-02-02 00:00:00     0.001234     0.005322     0.018439   
75%      2022-06-03 00:00:00     0.023860     0.053766     0.097590   
max      2026-10-02 00:00:00     0.699968     0.737780     0.955557   
std                      NaN     0.050354     0.096894     0.159484   

feature      vol_12w       target  
count    4465.000000  4520.000000  
mean        0.042514     0.001829  
min         0.007624    -0.293098  
25%         0.024395    -0.022136  
50%         0.035193     0.001234  
75%         0.051718     0.023860  
max         0.249565     0.699968  
std         0.027741     0.050354  
feature       week    commodity  ret

## Merge the prices data with the text one

In [42]:
names = list(tickers.values())
tone = weekly.melt(id_vars="week", value_vars=names, var_name="commodity", value_name="net_tone")
cnt = weekly.melt(id_vars="week", value_vars=[f"{c}_n" for c in names],
                  var_name="commodity", value_name="n_relevant")
cnt["commodity"] = cnt["commodity"].str.removesuffix("_n")
text_long = tone.merge(cnt, on=["week", "commodity"])

for df in (price_panel, text_long):
    df["week"] = pd.to_datetime(df["week"]).astype("datetime64[ns]")

panel = price_panel.merge(text_long, on=["week", "commodity"], how="left")
panel["net_tone"] = panel["net_tone"].fillna(0)
panel["n_relevant"] = panel["n_relevant"].fillna(0)
panel["has_news"] = (panel["n_relevant"] > 0).astype(int)
panel["attention"] = np.log1p(panel["n_relevant"])

panel = panel[(panel["week"] >= weekly["week"].min()) & (panel["week"] <= weekly["week"].max())]
panel = panel.dropna(subset=["mom_12w", "vol_12w", "target"]).reset_index(drop=True)
panel.to_csv("panel_weekly.csv", index=False)
print(panel.shape)

(4365, 11)


In [1]:
print(panel.describe())
print(panel.head())
print(panel.columns)
#(panel.isna()==True).sum()


#To keep all the files in one 
all_scores = pd.concat([pd.read_csv(f) for f in glob.glob("scores_*.csv")], ignore_index=True)
all_scores.to_csv("llm_scores_all.csv", index=False)

NameError: name 'panel' is not defined

## Feature construction

The final dataset (`panel_weekly.csv`) has **one row per week × commodity**, for five commodity
futures: crude oil (WTI), natural gas, gold, copper and wheat. Weeks run from Saturday to Friday
and are labelled by their **Friday** date. All features of week $t$ use only information available
at the Friday close of week $t$; the target is the return of the following week.

### 1. Text data → weekly text features

1. **Collection and cleaning.** News and policy texts were collected, dated, restricted to the study
   period and grouped by week (Friday label).
2. **Back to article level.** Each week's texts were split back into individual articles, so that
   each article can be scored separately.
3. **Relevance filter.** Only articles containing keywords related to monetary policy, fiscal policy,
   the macroeconomy or one of the five commodities were kept (e.g. "federal reserve", "inflation",
   "deficit", "crude", "OPEC", "natural gas", "copper", "wheat").
4. **LLM scoring.** Each article (first 2,000 characters) was scored with `gpt-4o-mini`
   (temperature 0, structured output). For each commodity $i$, the model returns an expected impact
   on its futures price over the next week:
   $s_{a,i} \in [-1, 1]$, from strongly bearish (−1) to strongly bullish (+1), with exactly 0 when
   the article is irrelevant to that commodity.
5. **Weekly aggregation.** For commodity $i$ and week $t$, only articles with $s_{a,i} \neq 0$ are
   considered *relevant*.

| Feature | Definition |
|---|---|
| `net_tone` | Mean impact over the relevant articles of the week: $\text{net\_tone}_{i,t} = \frac{1}{N_{i,t}}\sum_{a \text{ relevant}} s_{a,i}$; set to 0 if no relevant article |
| `n_relevant` | $N_{i,t}$, number of relevant articles for commodity $i$ in week $t$ |
| `has_news` | 1 if $N_{i,t} > 0$, else 0 (distinguishes "no news" from "neutral news") |
| `attention` | $\log(1 + N_{i,t})$, media attention, log-scaled to reduce the weight of very busy weeks |

### 2. Price data → weekly price features

1. **Prices.** Daily closing prices of continuous front-month futures from Yahoo Finance:
   `CL=F` (WTI), `NG=F` (natural gas), `GC=F` (gold), `HG=F` (copper), `ZW=F` (wheat).
2. **Weekly frequency.** The last close of each week (normally Friday) is kept: $P_{i,t}$.
3. **Weekly return.** $r_{i,t} = P_{i,t} / P_{i,t-1} - 1$. Returns are capped at ±50% to neutralise
   jumps caused by contract rolls or data errors.

| Feature | Definition |
|---|---|
| `ret` | Return of the current week: $r_{i,t}$ |
| `mom_4w` | 1-month momentum: $\sum_{k=0}^{3} r_{i,t-k}$ |
| `mom_12w` | 3-month momentum: $\sum_{k=0}^{11} r_{i,t-k}$ |
| `vol_12w` | Volatility: standard deviation of $r_{i,t-11}, \dots, r_{i,t}$ (weekly, not annualised) |

### 3. Target

| Variable | Definition |
|---|---|
| `target` | Next week's return: $r_{i,t+1}$ |

The target is never used as a feature: at the Friday close of week $t$, only `target` is unknown.

### 4. Merge

Price features (available every week) and text features were merged on (`week`, `commodity`).
Weeks without relevant news keep their price features, with `net_tone = 0`, `n_relevant = 0`,
`has_news = 0`, `attention = 0`. Rows without enough price history for `mom_12w` and `vol_12w`,
or without a next-week return, were removed.

### 5. Limitations

- **Look-ahead risk of the LLM:** `gpt-4o-mini` was trained on data up to about September 2023, so
  it may partly "know" what happened after older news. Results are therefore also reported on the
  period after October 2023.
- **Publication time:** articles are assigned to weeks by date only; an article published on Friday
  after the market close is counted in that week although it could only be traded the next week.
- **Continuous futures:** Yahoo Finance front-month series are not adjusted for contract rolls;
  the ±50% cap only removes the largest artefacts.

**Why there are many `scores_*.csv` files.** LLM scoring was run in blocks of articles rather than
all at once. Each block is saved in its own file as soon as it is scored (`scores_0.csv` for the
first block, `scores_100.csv` for the next, and so on, the number being the position of the block's
first article). This makes the process robust and cheap to rerun: if the run is interrupted
(network error, API limit, manual stop), only the unfinished block is lost, and on the next run the
blocks already saved are skipped instead of being sent to the API again. Each article is therefore
scored, and paid for, only once. The files are then combined into a single table before the weekly
aggregation.